In [13]:
# ============================================================
# Cell 1 — Colab Setup
# Clone repo into runtime and verify structure
# ============================================================

import os

# --- Environment Detection ---
COLAB = False  # Set to True if running in Google Colab

if COLAB:
    REPO_URL    = "https://github.com/manuelarceaguirre/capstone-quantum.git"
    BRANCH      = "David"  # change to your branch name
    REPO_ROOT   = "/content/capstone-quantum"

    %cd /content

    # remove stale runtime copy if it exists
    !rm -rf capstone-quantum

    # clone and checkout branch
    !git clone {REPO_URL}
    %cd /content/capstone-quantum
    !git checkout {BRANCH}
else:
    # Local environment — use repo root from current working directory
    REPO_ROOT = os.getcwd()

# verify expected directories exist
for d in ["data/processed", "notebooks", "results", "src"]:
    path = os.path.join(REPO_ROOT, d)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  {d:<25} {status}")

# create results directory if it doesn't exist yet
os.makedirs(os.path.join(REPO_ROOT, "results"), exist_ok=True)

print("\nSetup complete.")

  data/processed            OK
  notebooks                 OK
  results                   OK
  src                       OK

Setup complete.


In [14]:
# ============================================================
# Cell 2 — Imports
# ============================================================

import sys
import pickle
import time
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    median_absolute_error,
)

warnings.filterwarnings("ignore")

# add repo root to path so src/ is importable
sys.path.insert(0, REPO_ROOT)

print("Imports complete.")

Imports complete.


In [15]:
# ============================================================
# Cell 3 — Configuration
# ============================================================

# --- Paths ---
DATA_DIR    = Path(REPO_ROOT) / "data" / "processed"
RESULTS_DIR = Path(REPO_ROOT) / "results"

# --- Window Parameters ---
WINDOW_MONTHS = 120   # training window length
TEST_MONTHS   = 1     # test window length
STEP_MONTHS   = 1     # fold step size

# --- Evaluation Parameters ---
HORIZON_LIST  = [1, 3, 6]
TARGET_LIST   = ["S&P 500", "INDPRO", "PAYEMS", "S&P 500", "CPIAUCSL"]
ACTIVE_TRACKS = ["B"] #["A", "B", "D"]  # C tabled

# --- Track Registry ---
TRACK_CONFIG = {
    "A":      {targ : {"type": "dataframe", "path": DATA_DIR / "track_A_full.parquet"} for targ in TARGET_LIST},
    "B":      {"INDPRO": {"type": "dataframe", "path": DATA_DIR / "track_b_INDPRO.parquet"},
               "PAYEMS" : {"type": "dataframe", "path": DATA_DIR / "track_b_PAYEMS.parquet"},
               "CPIAUCSL"     : {"type": "dataframe", "path": DATA_DIR / "track_b_CPIAUCSL.parquet"},
               "S&P 500"      : {"type": "dataframe", "path": DATA_DIR / "track_b_S&P 500.parquet"},
         },     # tabled
    "D":      {targ: {"type": "pipeline", "path": DATA_DIR / "track_D_pipe.pkl"} for targ in TARGET_LIST},
    "D_mini": {targ: {"type": "pipeline", "path": DATA_DIR / "track_D_mini_pipe.pkl"} for targ in TARGET_LIST},
}

# --- Target Column Map ---
# maps (target, horizon) -> column name in targets.parquet
TARGET_COL_MAP = {
    (target, h): f"y_{target}_h{h}"
    for target in TARGET_LIST
    for h in HORIZON_LIST
}

print("Configuration set.")
print(f"  Window:  {WINDOW_MONTHS}m train / {TEST_MONTHS}m test / {STEP_MONTHS}m step")
print(f"  Targets: {TARGET_LIST}")
print(f"  Horizons: {HORIZON_LIST}")
print(f"  Active tracks: {ACTIVE_TRACKS}")

Configuration set.
  Window:  120m train / 1m test / 1m step
  Targets: ['S&P 500', 'INDPRO', 'PAYEMS', 'S&P 500', 'CPIAUCSL']
  Horizons: [1, 3, 6]
  Active tracks: ['B']


In [16]:
# ============================================================
# Cell 4 — Data Loading
# ============================================================

# --- Load track DataFrames ---
track_data = {}

for track_name, dict in TRACK_CONFIG.items():
    if track_name not in ACTIVE_TRACKS:
            print(f"  Track {track_name:<8} SKIPPED (inactive)")
            continue
    else:
        track_data[track_name] = {} 
        for target_name, config in dict.items():
            if target_name not in TARGET_LIST:
                print(f"  Target {target_name:<8} SKIPPED (inactive)")
                continue
            if config["type"] == "dataframe":
                track_data[track_name][target_name] = pd.read_parquet(config["path"])
                print(f"  Track {track_name:<8}, target {target_name:<8} loaded   shape={track_data[track_name][target_name].shape}")
            elif config["type"] == "pipeline":
                with open(config["path"], "rb") as f:
                    track_data[track_name][target_name] = pickle.load(f)
                print(f"  Track {track_name:<8}, target {target_name:<8} loaded   pipeline={type(track_data[track_name][target_name]).__name__}")

# --- Load stationary panel (needed for pipeline tracks) ---
stationary_panel = pd.read_parquet(DATA_DIR / "stationary_panel.parquet")
print(f"\n  Stationary panel loaded   shape={stationary_panel.shape}")

# --- Load targets ---
targets = pd.read_parquet(DATA_DIR / "targets.parquet")
print(f"  Targets loaded            shape={targets.shape}")

# --- Load USREC ---
usrec = pd.read_parquet(DATA_DIR / "usrec.parquet")
print(f"  USREC loaded              shape={usrec.shape}")

print("\nData loading complete.")

  Track A        SKIPPED (inactive)
  Track B       , target INDPRO   loaded   shape=(767, 10)
  Track B       , target PAYEMS   loaded   shape=(767, 10)
  Track B       , target CPIAUCSL loaded   shape=(767, 12)
  Track B       , target S&P 500  loaded   shape=(767, 12)
  Track D        SKIPPED (inactive)
  Track D_mini   SKIPPED (inactive)

  Stationary panel loaded   shape=(767, 123)
  Targets loaded            shape=(806, 20)
  USREC loaded              shape=(806, 1)

Data loading complete.


In [17]:
print("First track start:", track_data[ACTIVE_TRACKS[0]][TARGET_LIST[0]].index.min())
print("First track end:  ", track_data[ACTIVE_TRACKS[0]][TARGET_LIST[0]].index.max())

First track start: 1962-04-01 00:00:00
First track end:   2026-02-01 00:00:00


In [18]:
for name in ACTIVE_TRACKS:
    if isinstance(track_data[name], pd.DataFrame):
        print(f"  Track {name:<8} {track_data[name].index.min()} -> {track_data[name].index.max()}")
print(f"  Stationary   {stationary_panel.index.min()} -> {stationary_panel.index.max()}")

  Stationary   1962-04-01 00:00:00 -> 2026-02-01 00:00:00


In [19]:
def generate_folds(index, window_months, test_months, step_months):
    """
    Generate fixed sliding window pseudo-OOS folds.

    Parameters
    ----------
    index         : pd.DatetimeIndex — full date index of the panel
    window_months : int — training window length in months
    test_months   : int — test window length in months
    step_months   : int — step size between folds in months

    Returns
    -------
    folds : list of dict with train/test date boundaries
    """
    folds = []
    n = len(index)

    for start in range(0, n - window_months - test_months + 1, step_months):
        train_start = index[start]
        train_end   = index[start + window_months - 1]
        test_start  = index[start + window_months]
        test_end    = index[start + window_months + test_months - 1]

        folds.append({
            "fold_id":     len(folds),
            "train_start": train_start,
            "train_end":   train_end,
            "test_start":  test_start,
            "test_end":    test_end,
        })

    return folds

# generate folds using track A index as reference
folds = generate_folds(
    index         = track_data[ACTIVE_TRACKS[0]][TARGET_LIST[0]].index,
    window_months = WINDOW_MONTHS,
    test_months   = TEST_MONTHS,
    step_months   = STEP_MONTHS,
)

print(f"  Total folds: {len(folds)}")
print(f"\n  First fold:")
print(f"    train: {folds[0]['train_start'].date()} -> {folds[0]['train_end'].date()}")
print(f"    test:  {folds[0]['test_start'].date()}  -> {folds[0]['test_end'].date()}")
print(f"\n  Last fold:")
print(f"    train: {folds[-1]['train_start'].date()} -> {folds[-1]['train_end'].date()}")
print(f"    test:  {folds[-1]['test_start'].date()}  -> {folds[-1]['test_end'].date()}")

  Total folds: 647

  First fold:
    train: 1962-04-01 -> 1972-03-01
    test:  1972-04-01  -> 1972-04-01

  Last fold:
    train: 2016-02-01 -> 2026-01-01
    test:  2026-02-01  -> 2026-02-01


In [20]:
def compute_metrics(y_true, y_pred, naive_mae):
    """
    Compute all evaluation metrics.

    Parameters
    ----------
    y_true  : array-like — actual target values from test window
    y_pred  : array-like — model predictions
    naive_mae : float — MAE of naive baseline model on test set

    Returns
    -------
    dict of metric name -> float
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mae       = mean_absolute_error(y_true, y_pred)
    mse       = mean_squared_error(y_true, y_pred)
    rmse      = np.sqrt(mse)
    median_ae = median_absolute_error(y_true, y_pred)

    # avoid division by zero on flat training series
    if naive_mae == 0 or np.isnan(naive_mae):
        mase  = np.nan
    else:
        mase  = mae / naive_mae

    # RMSPE: Root Mean Squared Percentage Error
    with np.errstate(divide='ignore', invalid='ignore'):
        percentage_errors = ((y_true - y_pred) / y_true) ** 2
        percentage_errors = percentage_errors[np.isfinite(percentage_errors)]
        rmspe = np.sqrt(np.mean(percentage_errors)) if len(percentage_errors) > 0 else np.nan

    return {
        "mse":       mse,
        "mae":       mae,
        "median_ae": median_ae,
        "rmse":      rmse,
        "mase":      mase,
        "rmspe":     rmspe,
    }

print("Metric functions defined.")

Metric functions defined.


In [21]:
from src.model_functions import (arima_model_function,
                                 AR6_model_function,
                                 random_walk_model_function
)


# ============================================================
# Cell 7 — Model Registry
# ============================================================

# --- Example model for loop validation ---
def example_model(train_df, test_df, y_train, horizon):
    """
    Returns predictions as a pandas Series with test_df index.
    Used to validate loop mechanics before real models are integrated.
    
    Parameters
    ----------
    train_df : pd.DataFrame — training features
    test_df : pd.DataFrame — test features
    y_train : pd.Series — training target
    horizon : int — forecast horizon
    
    Returns
    -------
    pd.Series with index matching test_df.index
    """
    preds = np.random.randn(len(test_df))
    return pd.Series(preds, index=test_df.index)


# --- Registry ---
# key   : model name (used in results table)
# value : callable with signature (train_df, test_df, y_train, horizon)
#         -> pd.Series with index matching test_df.index

MODEL_REGISTRY = {
    #    "example": example_model,
    "arima": arima_model_function,
    "AR6": AR6_model_function,
    "random_walk": random_walk_model_function,
}

print(f"Models registered: {list(MODEL_REGISTRY.keys())}")


Models registered: ['arima', 'AR6', 'random_walk']


In [22]:

# ============================================================
# Cell 8 — Pipeline Helper
# Fit/transform pipeline tracks before passing to models
# ============================================================

def prepare_track(track_name, fold, target):
    """
    Slice and prepare train/test DataFrames for a given track and fold.

    For DataFrame tracks (A, B): slice directly by date.
    For Pipeline tracks (D, D_mini): slice stationary panel by date,
        fit pipeline on train only, transform both, return DataFrames.

    Parameters
    ----------
    track_name : str — one of ACTIVE_TRACKS
    fold       : dict — fold definition with train/test date boundaries

    Returns
    -------
    train_df : pd.DataFrame
    test_df  : pd.DataFrame
    """
    config = TRACK_CONFIG[track_name][target]

    if config["type"] == "dataframe":
        df = track_data[track_name][target]
        train_df = df.loc[fold["train_start"]:fold["train_end"]]
        test_df  = df.loc[fold["test_start"]:fold["test_end"]]

    elif config["type"] == "pipeline":
        # slice stationary panel
        train_raw = stationary_panel.loc[fold["train_start"]:fold["train_end"]]
        test_raw  = stationary_panel.loc[fold["test_start"]:fold["test_end"]]

        # fit on train only — no leakage
        pipeline = pickle.loads(pickle.dumps(track_data[track_name][target]))  # fresh copy
        pipeline.fit(train_raw)

        # transform both
        train_transformed = pipeline.transform(train_raw)
        test_transformed  = pipeline.transform(test_raw)

        # return as DataFrames preserving date index
        n_cols = train_transformed.shape[1]
        cols   = [f"f{i}" for i in range(n_cols)]

        train_df = pd.DataFrame(train_transformed, index=train_raw.index, columns=cols)
        test_df  = pd.DataFrame(test_transformed,  index=test_raw.index,  columns=cols)

    return train_df, test_df

print("Pipeline helper defined.")

Pipeline helper defined.


In [23]:
# ============================================================
# Cell 9 — Evaluation Loop
# ============================================================

results = []

total_runs = len(folds) * len(ACTIVE_TRACKS) * len(TARGET_LIST) * len(HORIZON_LIST) * len(MODEL_REGISTRY)
completed  = 0

for fold in folds:
    for track_name in ACTIVE_TRACKS:

        for target in TARGET_LIST:
            # slice and prepare track data once per fold/track
            train_df, test_df = prepare_track(track_name, fold, target)

            for horizon in HORIZON_LIST:

                target_col = TARGET_COL_MAP[(target, horizon)]

                # slice target series
                y_train = targets.loc[train_df.index, target_col]
                y_test  = targets.loc[test_df.index,  target_col]

                # NaN guard — skip if target unavailable
                if y_test.isna().any() or y_train.isna().any():
                    continue
                
                # Compute naive baseline MAE once per fold/track/target/horizon
                try:
                    y_naive_pred = y_train.shift(horizon).dropna()
                    y_train_aligned = y_train[y_naive_pred.index]
                    naive_mae = mean_absolute_error(y_train_aligned, y_naive_pred)
                except:
                    naive_mae = np.nan

                for model_name, model_fn in MODEL_REGISTRY.items():

                    row = {
                        "model":       model_name,
                        "track":       track_name,
                        "target":      target,
                        "horizon":     horizon,
                        "fold_id":     fold["fold_id"],
                        "train_start": fold["train_start"],
                        "train_end":   fold["train_end"],
                        "test_start":  fold["test_start"],
                        "test_end":    fold["test_end"],
                        "error":       None,
                    }

                    try:
            
                        t0 = time.time()
                        y_pred = model_fn(train_df, test_df, y_train, horizon)
                        row["runtime_sec"] = round(time.time() - t0, 4)
                        metrics = compute_metrics(y_test.values, y_pred.values, naive_mae)
                        row.update(metrics)
                    except Exception as e:
                        row["runtime_sec"] = np.nan
                        row["mse"]         = np.nan
                        row["mae"]         = np.nan
                        row["median_ae"]   = np.nan
                        row["rmse"]        = np.nan
                        row["mase"]        = np.nan
                        row["rmspe"]       = np.nan
                        row["error"]       = str(e)

                    results.append(row)
                    completed += 1

                    if completed % 1000 == 0:
                        print(f"  {completed}/{total_runs} runs complete")

print(f"\nLoop complete. Total rows: {len(results)}")

  1000/29115 runs complete
  2000/29115 runs complete
  3000/29115 runs complete
  4000/29115 runs complete
  5000/29115 runs complete
  6000/29115 runs complete
  7000/29115 runs complete
  8000/29115 runs complete
  9000/29115 runs complete
  10000/29115 runs complete
  11000/29115 runs complete
  12000/29115 runs complete
  13000/29115 runs complete
  14000/29115 runs complete
  15000/29115 runs complete
  16000/29115 runs complete
  17000/29115 runs complete
  18000/29115 runs complete
  19000/29115 runs complete
  20000/29115 runs complete
  21000/29115 runs complete
  22000/29115 runs complete
  23000/29115 runs complete
  24000/29115 runs complete
  25000/29115 runs complete
  26000/29115 runs complete
  27000/29115 runs complete
  28000/29115 runs complete

Loop complete. Total rows: 28965


In [24]:
# ============================================================
# Cell 10 — Save Results
# ============================================================

results_df = pd.DataFrame(results)

# --- Save per model ---
for model_name in results_df["model"].unique():
    model_df = results_df[results_df["model"] == model_name]
    out_path = RESULTS_DIR / f"{model_name}_raw.parquet"
    model_df.to_parquet(out_path, index=False)
    print(f"  Saved: {out_path.name}   rows={len(model_df)}")

# --- Summary table ---
summary = (
    results_df
    .groupby(["model", "track", "target", "horizon"])
    [["mse", "mae", "median_ae", "rmse", "mase", "rmspe"]]
    .mean()
    .round(6)
)

print("\nSummary — mean metrics across folds:")
print(summary.to_string())

  Saved: arima_raw.parquet   rows=9655
  Saved: AR6_raw.parquet   rows=9655
  Saved: random_walk_raw.parquet   rows=9655

Summary — mean metrics across folds:
                                         mse       mae  median_ae      rmse      mase      rmspe
model       track target   horizon                                                              
AR6         B     CPIAUCSL 1        0.000006  0.001775   0.001775  0.001775  0.938839   1.347292
                           3        0.000048  0.004929   0.004929  0.004929  1.065524   2.409017
                           6        0.000126  0.007790   0.007790  0.007790  1.114248   1.138889
                  INDPRO   1        0.000201  0.006012   0.006012  0.006012  0.941258   4.458508
                           3        0.000812  0.013029   0.013029  0.013029  1.087525   4.558850
                           6        0.001641  0.022739   0.022739  0.022739  1.102796   5.757726
                  PAYEMS   1        0.004969  0.006060   0.006060

In [25]:
summary

mse       mae  median_ae      rmse  \
model       track target   horizon                                            
AR6         B     CPIAUCSL 1        0.000006  0.001775   0.001775  0.001775   
                           3        0.000048  0.004929   0.004929  0.004929   
                           6        0.000126  0.007790   0.007790  0.007790   
                  INDPRO   1        0.000201  0.006012   0.006012  0.006012   
                           3        0.000812  0.013029   0.013029  0.013029   
                           6        0.001641  0.022739   0.022739  0.022739   
                  PAYEMS   1        0.004969  0.006060   0.006060  0.006060   
                           3        0.004798  0.007422   0.007422  0.007422   
                           6        0.004549  0.010375   0.010375  0.010375   
                  S&P 500  1        0.001375  0.026846   0.026846  0.026846   
                           3        0.008036  0.065167   0.065167  0.065167   
                           6        0.019977  0.100404   0.100404  0.100404   
arima       B     CPIAUCSL 1        0.000006  0.001715   0.001715  0.001715   
                           3        0.000050  0.005001   0.005001  0.005001   
                           6        0.000134  0.007978   0.007978  0.007978   
                  INDPRO   1        0.000154  0.005822   0.005822  0.005822   
                           3        0.000795  0.013232   0.013232  0.013232   
                           6        0.001609  0.022933   0.022933  0.022933   
                  PAYEMS   1        0.000335  0.002267   0.002267  0.002267   
                           3        0.000601  0.004356   0.004356  0.004356   
                           6        0.000860  0.008094   0.008094  0.008094   
                  S&P 500  1        0.001400  0.027103   0.027103  0.027103   
                           3        0.009603  0.071354   0.071354  0.071354   
                           6        0.021884  0.106747   0.106747  0.106747   
random_walk B     CPIAUCSL 1        0.000008  0.001950   0.001950  0.001950   
                           3        0.000054  0.005038   0.005038  0.005038   
                           6        0.000132  0.007890   0.007890  0.007890   
                  INDPRO   1        0.000139  0.006693   0.006693  0.006693   
                           3        0.000652  0.013252   0.013252  0.013252   
                           6        0.001424  0.022412   0.022412  0.022412   
                  PAYEMS   1        0.000075  0.001639   0.001639  0.001639   
                           3        0.000253  0.003916   0.003916  0.003916   
                           6        0.000411  0.007583   0.007583  0.007583   
                  S&P 500  1        0.002099  0.033552   0.033552  0.033552   
                           3        0.009329  0.071455   0.071455  0.071455   
                           6        0.021378  0.105361   0.105361  0.105361   

                                        mase      rmspe  
model       track target   horizon                       
AR6         B     CPIAUCSL 1        0.938839   1.347292  
                           3        1.065524   2.409017  
                           6        1.114248   1.138889  
                  INDPRO   1        0.941258   4.458508  
                           3        1.087525   4.558850  
                           6        1.102796   5.757726  
                  PAYEMS   1        2.971897   1.887893  
                           3        1.995267   1.505647  
                           6        1.652755   2.450741  
                  S&P 500  1        0.817812   7.382537  
                           3        0.943558   2.608037  
                           6        0.979869   3.684958  
arima       B     CPIAUCSL 1        0.909068   1.466377  
                           3        1.081178   2.441829  
                           6        1.136633   1.165883  
                  INDPRO   1        0.919221   3.733

In [26]:
# ============================================================
# Cell 11 — USREC Disaggregation
# Join recession indicator for recession vs expansion analysis
# ============================================================

# rename for clean merge
usrec_col = usrec.columns[0]
usrec_clean = usrec.rename(columns={usrec_col: "usrec"})

# join on test_start date
results_df = results_df.merge(
    usrec_clean,
    left_on="test_start",
    right_index=True,
    how="left"
)

# disaggregated summary
summary_disagg = (
    results_df
    .groupby(["model", "track", "target", "horizon", "usrec"])
    [["mse", "mae", "median_ae", "rmse", "mase", "rmspe"]]
    .mean()
    .round(6)
)

print("Disaggregated summary — recession (1) vs expansion (0):")
print(summary_disagg.to_string())

Disaggregated summary — recession (1) vs expansion (0):
                                               mse       mae  median_ae      rmse       mase      rmspe
model       track target   horizon usrec                                                               
AR6         B     CPIAUCSL 1       0      0.000005  0.001586   0.001586  0.001586   0.860288   1.348996
                                   1      0.000019  0.003242   0.003242  0.003242   1.546022   1.333686
                           3       0      0.000034  0.004327   0.004327  0.004327   0.956217   2.514833
                                   1      0.000152  0.009566   0.009566  0.009566   1.907481   1.571305
                           6       0      0.000074  0.006462   0.006462  0.006462   0.938967   1.027174
                                   1      0.000523  0.017967   0.017967  0.017967   2.457280   1.993362
                  INDPRO   1       0      0.000105  0.005132   0.005132  0.005132   0.793498   4.093419
        

In [28]:
# ============================================================
# Cell 12 — Export Summary to CSV
# ============================================================

# Save disaggregated summary to CSV
csv_path = RESULTS_DIR / "summary_disaggregated.csv"
summary_disagg.to_csv(csv_path)
print(f"Disaggregated summary saved to: {csv_path}")

Disaggregated summary saved to: c:\dev\projects\capstone-quantum\results\summary_disaggregated.csv
